In [7]:
from tensorflow.keras import layers
from tensorflow.keras.layers import TimeDistributed, LayerNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2

from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Conv2D,MaxPool2D,Flatten,LSTM, Dropout, Dense, TimeDistributed
import kapre
from kapre.composed import get_melspectrogram_layer
import tensorflow as tf
import os
from python_speech_features import mfcc
import pandas as pd
import numpy as np

import librosa
from tqdm import tqdm
from sklearn.utils.class_weight import compute_class_weight
from scipy.io import wavfile

In [26]:
data_dir = "../../../RAVDESS"
save_dir = "../../dataset/"
clean_dir = save_dir+"clean/"

def get_conv_model():
    model = Sequential()
    model.add(Conv2D(16,(3,3),activation="relu",strides=(1,1),
                    padding="same",input_shape=input_shape))
    model.add(Conv2D(32,(3,3),activation="relu",strides=(1,1),
             padding="same"))
    model.add(Conv2D(64,(3,3),activation="relu",strides=(1,1),
             padding="same"))
    model.add(Conv2D(128,(3,3),activation="relu",strides=(1,1),
             padding="same"))
    model.add(MaxPool2D((2,2)))
    model.add(Dropout(0.5))
    model.add(Flatten())
    model.add(Dense(128,activation="relu"))
    model.add(Dense(64,activation="relu"))
    model.add(Dense(8,activation="softmax"))
    print(model.summary())
    model.compile(loss="categorical_crossentropy",
                  optimizer="adam",
                 metrics=["acc"])
    return model

def get_recurrent_model():
    #shape of RNN is (n,time,feat)
    model = Sequential()
    model.add(LSTM(128,return_sequences=True,input_shape=input_shape))
    model.add(LSTM(128,return_sequences=True))
    model.add(Dropout(0.5))
    model.add(TimeDistributed(Dense(64,activation="relu")))
    model.add(TimeDistributed(Dense(32,activation="relu")))
    model.add(TimeDistributed(Dense(16,activation="relu")))
    model.add(TimeDistributed(Dense(8,activation="relu")))
    model.add(Flatten())
    model.add(Dense(8,activation="softmax"))
    print(model.summary())
    model.compile(loss="categorical_crossentropy",
                  optimizer="adam",
                 metrics=["acc"])
    return model

def build_rand_feat():
    X = []
    y = []
    _min, _max = float("inf"), float("-inf")
    
    for _ in tqdm(range(n_samples)):
        rand_class = np.random.choice(class_dist.index,p=prob_dist)
        file = np.random.choice(df[df["emotion"]==rand_class]["filename"])
        label = df[df["filename"]==file]["emotion"].iloc[0]
        rate,wav = wavfile.read(clean_dir+file)
        rand_index = np.random.randint(0,wav.shape[0]-config.step)
        
        sample = wav[rand_index:rand_index+config.step]
        X_sample = mfcc(sample,rate,numcep=config.nfeat,nfilt=config.nfilt,nfft=config.nfft)
        
        _min = min(np.amin(X_sample),_min)
        _max = max(np.amax(X_sample),_max)
        
        X.append(X_sample if config.mode == "conv" else X_sample.T)
        y.append(classes.index(label))
    X,y = np.array(X), np.array(y)
    X = (X - _min) / (_max - _min)
    
#     if config.mode == "conv":
#         X = X.reshape(X.shape[0],X.shape[1],X.shape[2],1)
#     elif config.mode == "time":
#         X = X.reshape(X.shape[0], X.shape[1],X.shape[2])
        
    y = to_categorical(y, num_classes=8)
    
    return X,y

In [12]:
df = pd.read_csv("audio_data.csv")
classes = list(df["emotion"].unique())
class_dist = df.groupby(["emotion"])["length"].mean()

prob_dist = class_dist/ class_dist.sum()
choices = np.random.choice(class_dist.index, p = prob_dist)

n_samples = 2 * int(df["length"].sum()/0.1)


class Config:
    def __init__(self,mode="conv",nfilt=26,nfeat=13,nfft=512,rate=16000):
        self.mode = mode
        self.nfilt = nfilt
        self.nfeat = nfeat
        self.nfft = nfft
        self.rate = rate
        self.step = int(rate/10)
        

config = Config()

if config.mode == "conv":
    X,y = build_rand_feat()
    y_flat = np.argmax(y,axis=1)
    input_shape = (X.shape[1],X.shape[2],1)
    model = get_conv_model()
elif config.mode == "time":
    X,y = build_rand_feat()
    y_flat = np.argmax(y,axis=1)
    input_shape = (X.shape[1],X.shape[2])
    model = get_recurrent_model()

# class_weight = compute_class_weight("balanced",np.unique(y_flat),y_flat)

model.fit(X,y,epochs=30,batch_size=32,
          shuffle=True)

100%|█████████████████████████████████████████████████████████████████████████| 106578/106578 [02:48<00:00, 631.86it/s]


IndexError: tuple index out of range

In [30]:
X2,y2 = build_rand_feat()

100%|█████████████████████████████████████████████████████████████████████████| 106578/106578 [04:03<00:00, 437.02it/s]


In [24]:
model = get_conv_model()
model.fit(X,y,epochs=30,batch_size=32,
          shuffle=True)

ValueError: Exception encountered when calling layer "max_pooling2d" (type MaxPooling2D).

Negative dimension size caused by subtracting 2 from 1 for '{{node max_pooling2d/MaxPool}} = MaxPool[T=DT_FLOAT, data_format="NHWC", explicit_paddings=[], ksize=[1, 2, 2, 1], padding="VALID", strides=[1, 2, 2, 1]](Placeholder)' with input shapes: [?,17,1,128].

Call arguments received by layer "max_pooling2d" (type MaxPooling2D):
  • inputs=tf.Tensor(shape=(None, 17, 1, 128), dtype=float32)

In [44]:
#     X,y = build_rand_feat()
#     y_flat = np.argmax(y,axis=1)
#     input_shape = (X.shape[1],X.shape[2],1)
#     model = get_conv_model()


X3 = X.reshape(X.shape[0],X.shape[1],1,1)
y_flat = np.argmax(y,axis=1)
input_shape = (X3.shape[1],X3.shape[2])
model = get_recurrent_model()
model.fit(X3,y,epochs=30,batch_size=32,
          shuffle=True)

Model: "sequential_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm (LSTM)                 (None, 17, 128)           66560     
                                                                 
 lstm_1 (LSTM)               (None, 17, 128)           131584    
                                                                 
 dropout (Dropout)           (None, 17, 128)           0         
                                                                 
 time_distributed (TimeDistr  (None, 17, 64)           8256      
 ibuted)                                                         
                                                                 
 time_distributed_1 (TimeDis  (None, 17, 32)           2080      
 tributed)                                                       
                                                                 
 time_distributed_2 (TimeDis  (None, 17, 16)          

KeyboardInterrupt: 

In [39]:
input_shape

(17, 1, 1)

In [29]:
X.shape

(106578, 17)

In [31]:
X2.shape

(106578, 9, 13)

In [32]:
X.shape

(106578, 17)

In [35]:
X2.reshape(X2.shape[0],X2.shape[1],X2.shape[2],1).shape

(106578, 9, 13, 1)

In [42]:
(X3.shape[1],X3.shape[2],1)

(17, 1, 1)

../../dataset/clean\01-01-01-01-01.wav


Error: unknown format: 3

In [49]:
!pip install audiolazy

In [59]:
wav.read(wav)

AttributeError: module 'scipy.io.wavfile' has no attribute 'seek'